# FinTrust LLM forecaster, experiments 2 and 3

Same setup as `kaggle_llm_forecaster.ipynb` (GPU T4 x2, internet on, `HF_TOKEN` secret, `nse_dataset` attached).
Protocol: addendum in `PREREGISTRATION.md`.
- Experiment 2: Llama 3.1 8B gets the conventional agents' signals and writes a short reason before its probability.
- Experiment 3: Qwen2.5-7B-Instruct with the experiment 1 prompts (different model family).

Run with *Save & Run All (Commit)*; roughly 5 to 7 hours (estimate, within Kaggle's 12-hour limit).
Download `llm_outputs_exp23.zip` from the Output tab when it finishes.

In [ ]:
import os, sys, glob, subprocess, hashlib, shutil, time
REPO = 'https://github.com/IAbdullahSlash/fintrust.git'
BRANCH = 'claude/project-thread-k1tvaj'          # falls back to the default branch once this one is merged
if not os.path.exists('fintrust'):
    if subprocess.call(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, 'fintrust']) != 0:
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO, 'fintrust'])
print(subprocess.check_output(['git', '-C', 'fintrust', 'log', '-1', '--format=%H %s']).decode())

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate yfinance
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Price data: the paper's copy if attached as a Kaggle Dataset, otherwise a fresh Yahoo Finance download
found = glob.glob('/kaggle/input/**/nse_daily.csv', recursive=True)
if found: PRICES = found[0]
else:
    subprocess.check_call([sys.executable, 'fintrust/data/download_nse.py']); PRICES = 'fintrust/data/raw/nse_daily.csv'
print(PRICES, hashlib.sha256(open(PRICES, 'rb').read()).hexdigest())

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login; login(token=os.environ['HF_TOKEN'])
sys.path.insert(0, 'fintrust/paper/experiments/llm')
import llm_forecaster as L
REQ, OUT = '/kaggle/working/requests', '/kaggle/working/outputs'
counts = L.build(PRICES, REQ, label_csv='fintrust/paper/experiments/fair/nse_vol_forecasts.csv')

In [ ]:
be = L.HFBackend('meta-llama/Llama-3.1-8B-Instruct', max_batch=16); print('revision', be.revision)
t0 = time.time(); p = L.run(be, f'{REQ}/feat_clean.jsonl', OUT, 'llama31_8b', limit=32)      # pilot
print(f'{time.time() - t0:.1f}s for 32'); print(open(p).read()[:2000])

In [ ]:
for cond in ['feat_clean', 'feat_news_alarm', 'feat_news_calm', 'feat_persona']:
    t0 = time.time(); L.run(be, f'{REQ}/{cond}.jsonl', OUT, 'llama31_8b'); print(cond, f'{(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
del be; import gc; gc.collect(); torch.cuda.empty_cache()
be = L.HFBackend('meta-llama/Llama-3.2-1B-Instruct', max_batch=32)
L.run(be, f'{REQ}/feat_clean.jsonl', OUT, 'llama32_1b', start='2025-01-01')

In [ ]:
# Experiment 3: Qwen2.5-7B-Instruct, experiment 1 prompts
del be; gc.collect(); torch.cuda.empty_cache()
be = L.HFBackend('Qwen/Qwen2.5-7B-Instruct', max_batch=16); print('revision', be.revision)
for cond in ['clean', 'news_alarm', 'news_calm', 'persona', 'clean_pre', 'named', 'named_pre']:
    t0 = time.time(); L.run(be, f'{REQ}/{cond}.jsonl', OUT, 'qwen25_7b'); print(cond, f'{(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
import json
for f in sorted(glob.glob(f'{OUT}/*.jsonl')):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), len(rows), 'parse failures:', sum(not r['parse_ok'] for r in rows), 'mean p: %.3f' % (sum(r['p'] for r in rows) / len(rows)))
shutil.make_archive('/kaggle/working/llm_outputs_exp23', 'zip', OUT); print('download llm_outputs_exp23.zip from the Output tab')